# 🛠️ Sysadmin Game: Train an LLM to Fix Linux Systems

This notebook trains a Qwen2.5-Coder model to diagnose and fix broken Linux containers using:
- **Phase 1**: SFT warm-start on 92 curated troubleshooting traces
- **Phase 2**: GRPO reinforcement learning with live container feedback

**Requirements**: GPU runtime (T4 minimum, A100 recommended)

---

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install Unsloth (optimized for Colab)
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
# Install other dependencies
%%capture
!pip install trl>=0.8.0 transformers>=4.40.0 datasets>=2.18.0 wandb>=0.16.0 matplotlib peft bitsandbytes

In [ ]:
# Clone the sysadmin-game repository
!git clone https://github.com/shettydevesh/sysadmin.git
%cd sysadmin

In [ ]:
# Verify dataset exists
!ls -la dataset/
!wc -l dataset/*.jsonl

## 2. Configuration

In [ ]:
# Training configuration
CONFIG = {
    # Model - use 3B for T4, 7B for A100
    "model_name": "Qwen/Qwen2.5-Coder-3B-Instruct",  # Change to 7B-Instruct for A100
    
    # LoRA settings
    "lora_r": 16,
    "lora_alpha": 16,
    
    # Training
    "max_seq_length": 2048,
    "epochs": 2,
    "batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    
    # Paths
    "train_data": "dataset/sft_train.jsonl",
    "val_data": "dataset/sft_val.jsonl",
    "output_dir": "checkpoints/sft",
    
    # Logging
    "use_wandb": False,  # Set True and login to enable
    "wandb_project": "sysadmin-game",
}

print("Configuration loaded!")
print(f"Model: {CONFIG['model_name']}")
print(f"Epochs: {CONFIG['epochs']}")

In [ ]:
# Optional: Login to W&B for experiment tracking
if CONFIG["use_wandb"]:
    import wandb
    wandb.login()
    wandb.init(project=CONFIG["wandb_project"], name="sft-training")

## 3. Load Dataset

In [ ]:
import json
from pathlib import Path

def load_jsonl(path):
    """Load JSONL dataset."""
    data = []
    with open(path) as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Load data
train_data = load_jsonl(CONFIG["train_data"])
val_data = load_jsonl(CONFIG["val_data"])

print(f"Training examples: {len(train_data)}")
print(f"Validation examples: {len(val_data)}")

# Show scenario distribution
from collections import Counter
scenarios = Counter([ex.get("scenario_id", "unknown") for ex in train_data])
print(f"\nScenarios: {dict(scenarios)}")

In [ ]:
# Preview a training example
import pprint

example = train_data[0]
print(f"Scenario: {example.get('scenario_id')}")
print(f"Number of turns: {len(example['messages'])}")
print("\n--- First few messages ---")
for msg in example["messages"][:4]:
    role = msg["role"]
    content = msg["content"][:200] + "..." if len(msg["content"]) > 200 else msg["content"]
    print(f"\n[{role}]\n{content}")

## 4. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

print(f"Loading model: {CONFIG['model_name']}")
print(f"Max sequence length: {CONFIG['max_seq_length']}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,  # Auto-detect
    load_in_4bit=True,
)

print(f"\nModel loaded! Device: {model.device}")

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 5. Prepare Training Data

In [ ]:
from datasets import Dataset

def format_for_training(examples):
    """Convert to {messages: [...]} format for SFT."""
    return [{"messages": ex["messages"]} for ex in examples]

def apply_chat_template(examples):
    """Apply tokenizer chat template."""
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

# Format data
train_formatted = format_for_training(train_data)
val_formatted = format_for_training(val_data)

# Create HF datasets
train_dataset = Dataset.from_list(train_formatted)
train_dataset = train_dataset.map(apply_chat_template, batched=True)

val_dataset = Dataset.from_list(val_formatted)
val_dataset = val_dataset.map(apply_chat_template, batched=True)

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Val dataset: {len(val_dataset)} examples")

In [ ]:
# Preview formatted example
print("--- Formatted training example (first 500 chars) ---")
print(train_dataset[0]["text"][:500])

## 6. Train with SFT

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    logging_steps=5,
    save_steps=25,
    eval_strategy="steps",
    eval_steps=25,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=True,
    report_to="wandb" if CONFIG["use_wandb"] else "none",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    tokenizer=tokenizer,
)

print("Trainer ready!")
print(f"Total training steps: {trainer.state.max_steps if hasattr(trainer.state, 'max_steps') else 'TBD'}")

In [ ]:
# Start training
print("Starting SFT training...")
print("="*50)

trainer_stats = trainer.train()

print("="*50)
print("Training complete!")
print(f"Total training time: {trainer_stats.metrics['train_runtime']:.2f}s")

In [ ]:
# Save the trained model
print(f"Saving model to {CONFIG['output_dir']}")
model.save_pretrained(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print("Model saved!")

## 7. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt
import os

# Create results directory
os.makedirs("results", exist_ok=True)

# Extract training history
history = trainer.state.log_history

train_losses = [(h["step"], h["loss"]) for h in history if "loss" in h and "eval_loss" not in h]
eval_losses = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

# Plot training curve
fig, ax = plt.subplots(figsize=(10, 6))

if train_losses:
    steps, losses = zip(*train_losses)
    ax.plot(steps, losses, label="Train Loss", alpha=0.8)

if eval_losses:
    steps, losses = zip(*eval_losses)
    ax.plot(steps, losses, label="Eval Loss", marker="o")

ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.set_title("SFT Training - Loss Curve")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/sft_loss_curve.png", dpi=150)
plt.show()

print("Saved to results/sft_loss_curve.png")

## 8. Test the Trained Model

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

# Test prompt - simulating a user complaint
test_messages = [
    {"role": "system", "content": "You are an SRE agent. Diagnose and fix Linux system issues. Use <think> for reasoning, <bash> for commands."},
    {"role": "user", "content": "nginx won't start. Help!"}
]

# Format input
input_text = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,
)

# Generate response
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract just the assistant response
print("--- Model Response ---")
print(response.split("assistant")[-1] if "assistant" in response else response[-500:])

In [ ]:
# Test with another scenario
test_messages_2 = [
    {"role": "system", "content": "You are an SRE agent. Diagnose and fix Linux system issues. Use <think> for reasoning, <bash> for commands."},
    {"role": "user", "content": "Disk is full and I can't write any files. What do I do?"}
]

input_text = tokenizer.apply_chat_template(
    test_messages_2,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("--- Model Response (Disk Full) ---")
print(response.split("assistant")[-1] if "assistant" in response else response[-500:])

## 9. Save to Hugging Face Hub (Optional)

In [ ]:
# Uncomment and run to upload to HF Hub

# from huggingface_hub import login
# login()

# HF_USERNAME = "your-username"  # Change this
# model.push_to_hub(f"{HF_USERNAME}/sysadmin-qwen-sft")
# tokenizer.push_to_hub(f"{HF_USERNAME}/sysadmin-qwen-sft")
# print(f"Model uploaded to https://huggingface.co/{HF_USERNAME}/sysadmin-qwen-sft")

## 10. Download Results

In [ ]:
# Zip results for download
!zip -r sysadmin_results.zip checkpoints/sft results/

# Download link will appear in Colab file browser
print("\nDownload sysadmin_results.zip from the file browser (left panel)")

---

## Next Steps

1. **GRPO Training**: Run `training/train_grpo.py` with the environment server for RL fine-tuning
2. **Evaluation**: Compare trained vs baseline on held-out scenarios
3. **Deploy**: Upload to HF Spaces as an interactive demo

See the [README](https://github.com/shettydevesh/sysadmin) for full documentation.